# kci 방식 오버랩 실헙 - 데이터셋을 변경하면서

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

ALPHA = 0.05    # 95% 신뢰구간
MAXQ = 500      # 루프 내 최대 쿼리 수

# ===현재 노트북 환경에 맞게 수정===
DATASETS = {
    "CardBase": {"path": Path("data/CardBase.csv"), "column": "Credit_Limit"},
    "Bank": {"path": Path("data/bank_exp.parquet"), "column": "credit_limit"},
}

# ===드라이브 전용===
# DATASETS = {
#     "CardBase": {"path": Path("/content/CardBase.csv"), "column": "Credit_Limit"},
#     "Bank": {"path": Path("/content/bank_exp.parquet"), "column": "credit_limit"},
# }

OUTPUT_DIR = Path("/content/outputs/result_single")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# @title
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
})

In [ ]:
# @title
def load_dataset(path, column):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"지원하지 않는 파일 형식: {path}")
    if column not in df.columns:
        raise KeyError(f"{path}에 {column!r} 열이 없습니다.")
    x = pd.to_numeric(df[column], errors="coerce").dropna().to_numpy(float)
    if len(x) == 0:
        raise ValueError(f"{path}의 {column!r} 열에 유효한 숫자가 없습니다.")
    n = len(x)
    return {
        "x": x,
        "n": n,
        "true_mean": float(x.mean()),
        "min": float(x.min()),
        "max": float(x.max()),
        "delta": float((x.max() - x.min()) / n),
    }

In [ ]:
DATA = {name: load_dataset(cfg["path"], cfg["column"])
        for name, cfg in DATASETS.items()}

# ===통계 정보를 DataFrame으로 변환===
pd.DataFrame([
    {"dataset": name, **{k: v for k, v in d.items() if k != "x"}}
    for name, d in DATA.items()
])

In [ ]:
# @title
def _statistics(values, true_mean, alpha=ALPHA):
    """현재까지의 응답값으로 누적 통계와 95% CI를 계산한다."""
    values = np.asarray(values, dtype=float)


    # 응답 값 개수로 k를 계산함
    k = len(values)
    cumulative_mean = float(values.mean())

    if k < 2:
        return {
            "k": k, "response": float(values[-1]),
            "cumulative_mean": cumulative_mean,
            "sample_std": np.nan, "standard_error": np.nan,
            "moe": np.nan, "ci_lo": np.nan, "ci_hi": np.nan,
            "ci_width": np.nan, "relative_width": np.nan,
            "hit": False, "covered": pd.NA,
        }

    sample_std = float(values.std(ddof=1))
    standard_error = sample_std / np.sqrt(k)
    t_value = stats.t.ppf(1 - alpha / 2, df=k - 1)
    moe = float(t_value * standard_error)
    ci_lo = cumulative_mean - moe
    ci_hi = cumulative_mean + moe
    ci_width = float(2 * moe)
    relative_width = float(ci_width / abs(cumulative_mean)) if cumulative_mean != 0 else np.inf
    covered = bool(ci_lo <= true_mean <= ci_hi)

    return {
        "k": k, "response": float(values[-1]),
        "cumulative_mean": cumulative_mean,
        "sample_std": sample_std, "standard_error": standard_error,
        "moe": moe, "ci_lo": ci_lo, "ci_hi": ci_hi,
        "ci_width": ci_width, "relative_width": relative_width,
        "hit": False, "covered": covered,
    }

def _final_from_history(history, **metadata):
    # 히스토리 데이터에서 마지막 행을 꺼내서, 메타데이터를 덧붙여 반환
    row = history.iloc[-1].to_dict()
    row.update(metadata)
    return row

def _validate_inputs(eps, threshold, streak_required, maxq):
    if eps <= 0:
        raise ValueError("eps는 0보다 커야 합니다.")
    if threshold <= 0:
        raise ValueError("threshold는 0보다 커야 합니다.")
    if streak_required < 1:
        raise ValueError("streak_required는 1 이상이어야 합니다.")
    if maxq < 2:
        raise ValueError("maxq는 2 이상이어야 합니다.")

> 오버랩 실험 유틸리티 함수
* `_overlap_from_ci()`: 원본 CI를 -delta만큼 이동한 가상 인접 CI와 비교

In [ ]:
def _overlap_from_ci(ci_lo, ci_hi, delta):
    """원본 CI를 -delta만큼 이동한 가상 인접 CI와 비교한다."""
    if not (np.isfinite(ci_lo) and np.isfinite(ci_hi)):
        return {"adj_ci_lo": np.nan, "adj_ci_hi": np.nan,
                "overlap_width": np.nan, "overlap_ratio": np.nan,
                "width_to_delta": np.nan}
    adj_ci_lo = ci_lo - delta
    adj_ci_hi = ci_hi - delta
    overlap_width = max(0.0, min(ci_hi, adj_ci_hi) - max(ci_lo, adj_ci_lo))
    ci_width = ci_hi - ci_lo
    return {
        "adj_ci_lo": adj_ci_lo,
        "adj_ci_hi": adj_ci_hi,
        "overlap_width": overlap_width,
        "overlap_ratio": overlap_width / ci_width if ci_width > 0 else np.nan,
        "width_to_delta": ci_width / delta if delta > 0 else np.nan,
    }

> kci에서 오버랩 실험&시각화 함수
* `run_kci_overlap_live()`
* `plot_kci_overlap_trajectory()`

In [ ]:
# @title
def run_kci_overlap_live(data, eps, threshold, streak_required, maxq=MAXQ,
                  seed=0, alpha=ALPHA, verbose=False):
    """KCI 방식의 단일 시행.

    질의를 순차적으로 수행하고 상대폭이 threshold 미만인 상태가
    streak_required회 연속이면 그 시점에서 중단한다.
    반환: (전체 질의 궤적 DataFrame, 최종 결과 dict)
    """
    _validate_inputs(eps, threshold, streak_required, maxq)
    rng = np.random.default_rng(seed)
    scale = data["delta"] / eps
    responses = []
    rows = []
    streak_count = 0
    stopped = False

    for k in range(1, maxq + 1):
        response = float(data["true_mean"] + rng.laplace(0.0, scale))
        responses.append(response)
        row = _statistics(responses, data["true_mean"], alpha)
        row.update(_overlap_from_ci(row["ci_lo"], row["ci_hi"], data["delta"]))
        hit = bool(np.isfinite(row["relative_width"]) and row["relative_width"] < threshold)
        streak_count = streak_count + 1 if hit else 0
        row.update({"threshold": threshold, "hit": hit, "streak_count": streak_count})
        rows.append(row)

        if verbose:
            print(f"k={k:4d} | response={response:,.3f} | mean={row['cumulative_mean']:,.3f} | "
                  f"width={row['ci_width'] if np.isfinite(row['ci_width']) else np.nan:,.3f} | "
                  f"relw={row['relative_width'] if np.isfinite(row['relative_width']) else np.nan:.6f} | "
                  f"hit={hit} | streak={streak_count}")

        if streak_count >= streak_required:
            stopped = True
            break

    history = pd.DataFrame(rows)
    final = _final_from_history(
        history,
        method="KCI",
        eps=float(eps),
        eps_total=float(eps * len(history)),
        threshold=float(threshold),
        streak_required=int(streak_required),
        seed=int(seed),
        delta=float(data["delta"]),
        laplace_scale=float(scale),
        true_mean=float(data["true_mean"]),
        k_stop=int(len(history)),
        converged=bool(stopped),
    )
    if not stopped:
        raise RuntimeError(f"KCI가 maxq={maxq}회 안에 수렴하지 않았습니다.")
    return history, final

In [ ]:
# @title
def _plot_stop_overlap(ax, history, final, blue="#2166AC", orange="#d6604d",
                       green="#1A9850", purple="#762A83", gray="#666666"):
    """수렴 시점의 원본 CI와 Δ 이동 인접 CI의 겹침을 수평 interval plot으로 그린다.

    KCI(final에 eps_total)와 INVERSE(final에 actual_eps_total)를 모두 지원.
    """
    delta = float(final["delta"])
    shift = -delta                       # 인접 DB 방향: 최대값 레코드 제거
    mean = float(final["cumulative_mean"])
    ci_lo, ci_hi = float(final["ci_lo"]), float(final["ci_hi"])
    adj_center, adj_lo, adj_hi = mean + shift, ci_lo + shift, ci_hi + shift
    ov_lo, ov_hi = max(ci_lo, adj_lo), min(ci_hi, adj_hi)
    ov_w = max(0.0, ov_hi - ov_lo)
    ov_ratio = ov_w / (ci_hi - ci_lo) if ci_hi > ci_lo else np.nan
    eps_label = final.get("eps_total", final.get("actual_eps_total"))

    y_resp = .02 * np.sin(np.arange(len(history)) * 2.4)
    ax.scatter(history["response"], y_resp, s=28, color=gray, alpha=.7, zorder=3)
    ax.axvline(final["true_mean"], color=green, ls="--", lw=1.5, label="True mean")
    ax.plot([ci_lo, ci_hi], [.62, .62], color=blue, lw=8, solid_capstyle="round",
            label="Original 95% CI")
    ax.plot([adj_lo, adj_hi], [.35, .35], color=orange, lw=8, solid_capstyle="round",
            label="Shifted adjacent 95% CI")
    if ov_w > 0:
        ax.plot([ov_lo, ov_hi], [.485, .485], color=purple, lw=10,
                solid_capstyle="round", label="CI overlap")
    ax.axvline(mean, color=blue, lw=1.2, alpha=.85)
    ax.axvline(adj_center, color=orange, lw=1.2, alpha=.85)
    mid = (mean + adj_center) / 2
    ax.annotate("", xy=(adj_center, .86), xytext=(mean, .86),
                arrowprops=dict(arrowstyle="<->", color="black", lw=1.2))
    ax.text(mid, .89, f"Δ={delta:,.1f}", ha="center", va="bottom", fontsize=9)
    ax.set_yticks([0, .35, .485, .62, .86])
    ax.set_yticklabels(["Responses", "Adjacent CI", "Overlap", "Original CI", "Centers"])
    ax.set_xlabel("Query output")
    ax.set_title("Stopping-point interval comparison")
    ax.set_ylim(-.12, 1.02)
    ax.legend(ncol=3, loc="upper left", bbox_to_anchor=(0, -0.22), frameon=False)
    ax.text(1.0, 1.02,
            f"k_stop={final['k_stop']} | ε_total={eps_label:.2f}\n"
            f"width={final['ci_width']:,.1f} | relw={final['relative_width']:.4f}\n"
            f"overlap ratio={ov_ratio:.3f} | width/Δ={final['ci_width']/delta:.3f}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=9,
            bbox=dict(boxstyle="round,pad=.35", facecolor="white", edgecolor="#bbbbbb"))

In [ ]:
# @title
def plot_kci_overlap_trajectory(history, final, title=None, save=None):
    """질의 증가에 따른 R_k, W_k, r_k + 중단 시점의 원본/인접 CI 겹침을 시각화한다."""
    h = history.copy()
    valid = h["k"] >= 2
    blue, red, green, purple = "#2166AC", "#D55E4A", "#1A9850", "#762A83"

    # 3개 곡선 패널(sharex) + 겹침 비교 패널(전체폭)을 분리한 gridspec
    fig = plt.figure(figsize=(11, 12.5), constrained_layout=True)
    gs = fig.add_gridspec(4, 1, height_ratios=[1, 1, 1, 1.15])
    axes = [fig.add_subplot(gs[i]) for i in range(3)]
    for ax in axes[1:]:
        ax.sharex(axes[0])
    ax4 = fig.add_subplot(gs[3])          # 겹침 패널은 x축이 독립 (query output 축)

    hit = valid & (h["hit"] == True)

    axes[0].plot(h["k"], h["relative_width"], color=blue, lw=1.8, label=r"Relative width $R_k$")
    axes[0].axhline(final["threshold"], color=red, ls="--", lw=1.2, label="Threshold")
    axes[0].scatter(h.loc[hit, "k"], h.loc[hit, "relative_width"], color=red, s=24, zorder=3, label="Threshold hit")
    axes[0].axvline(final["k_stop"], color="black", ls=":", lw=1.2, label=f"Stop k={final['k_stop']}")
    axes[0].set_ylabel(r"$R_k$"); axes[0].set_title("Relative CI width"); axes[0].legend(frameon=False, ncol=3)

    axes[1].plot(h["k"], h["ci_width"], color=purple, lw=1.8, label=r"Absolute CI width $W_k$")
    axes[1].axvline(final["k_stop"], color="black", ls=":", lw=1.2)
    axes[1].set_ylabel(r"$W_k$"); axes[1].set_title("Absolute confidence-interval width"); axes[1].legend(frameon=False)

    axes[2].plot(h["k"], h["overlap_ratio"], color=green, lw=1.8, label=r"Overlap ratio $r_k$")
    axes[2].axvline(final["k_stop"], color="black", ls=":", lw=1.2)
    axes[2].scatter([final["k_stop"]], [final["overlap_ratio"]], color=green, s=45, zorder=3, label="Stop-point overlap")
    axes[2].set_ylim(-0.05, 1.05); axes[2].set_ylabel(r"$r_k$"); axes[2].set_xlabel("Query count k")
    axes[2].set_title("CI overlap after shifting center by sensitivity"); axes[2].legend(frameon=False)

    # ── 4번째 행 (신규): 중단 시점 원본/인접 CI 겹침 interval 비교 ──
    _plot_stop_overlap(ax4, history, final)

    fig.suptitle(title or "KCI trajectory: relative width, CI width, and overlap", fontsize=14, fontweight="bold")
    if save: fig.savefig(save, dpi=300, bbox_inches="tight")
    plt.show()
    return fig

# CardBase에 대한 오버랩 실험

In [ ]:
history, final = run_kci_overlap_live(DATA["CardBase"],
                                      0.5, 0.01, 5, maxq=MAXQ,
                                      seed=5)

In [ ]:
plot_kci_overlap_trajectory(history, final, title=None, save=None);

# Bank에 대한 오버랩 실험

In [ ]:
bank_history, bank_final = run_kci_overlap_live(DATA["Bank"],
                                                0.5, 0.01, 5, maxq=MAXQ,
                                                seed=5)

In [ ]:
plot_kci_overlap_trajectory(bank_history, bank_final, title=None, save=None);